In [1]:
import pandas as pd

df_train_attacks = pd.read_csv("data/train_attacks.csv")
df_train_normal = pd.read_csv("data/train_normal.csv")
df_test = pd.read_csv("data/test.csv")

print("attacks\n", df_train_attacks.shape)
print("normal\n", df_train_normal.shape)
print("test\n", df_test.shape)

attacks
 (4177, 45)
normal
 (8761, 45)
test
 (2089, 44)


In [2]:
print("attacks\n", df_train_attacks.columns)
print("normal\n", df_train_normal.columns)
print("test\n", df_test.columns)

attacks
 Index(['DATETIME', ' L_T1', ' L_T2', ' L_T3', ' L_T4', ' L_T5', ' L_T6',
       ' L_T7', ' F_PU1', ' S_PU1', ' F_PU2', ' S_PU2', ' F_PU3', ' S_PU3',
       ' F_PU4', ' S_PU4', ' F_PU5', ' S_PU5', ' F_PU6', ' S_PU6', ' F_PU7',
       ' S_PU7', ' F_PU8', ' S_PU8', ' F_PU9', ' S_PU9', ' F_PU10', ' S_PU10',
       ' F_PU11', ' S_PU11', ' F_V2', ' S_V2', ' P_J280', ' P_J269', ' P_J300',
       ' P_J256', ' P_J289', ' P_J415', ' P_J302', ' P_J306', ' P_J307',
       ' P_J317', ' P_J14', ' P_J422', ' ATT_FLAG'],
      dtype='str')
normal
 Index(['DATETIME', 'L_T1', 'L_T2', 'L_T3', 'L_T4', 'L_T5', 'L_T6', 'L_T7',
       'F_PU1', 'S_PU1', 'F_PU2', 'S_PU2', 'F_PU3', 'S_PU3', 'F_PU4', 'S_PU4',
       'F_PU5', 'S_PU5', 'F_PU6', 'S_PU6', 'F_PU7', 'S_PU7', 'F_PU8', 'S_PU8',
       'F_PU9', 'S_PU9', 'F_PU10', 'S_PU10', 'F_PU11', 'S_PU11', 'F_V2',
       'S_V2', 'P_J280', 'P_J269', 'P_J300', 'P_J256', 'P_J289', 'P_J415',
       'P_J302', 'P_J306', 'P_J307', 'P_J317', 'P_J14', 'P_J422', 'ATT_F

In [3]:
print("attacks\n", df_train_attacks["DATETIME"].head(5))
print("normal\n", df_train_normal["DATETIME"].head(5))
print("test\n", df_test["DATETIME"].head(5))

attacks
 0    04/07/16 00
1    04/07/16 01
2    04/07/16 02
3    04/07/16 03
4    04/07/16 04
Name: DATETIME, dtype: str
normal
 0    06/01/14 00
1    06/01/14 01
2    06/01/14 02
3    06/01/14 03
4    06/01/14 04
Name: DATETIME, dtype: str
test
 0    04/01/17 00
1    04/01/17 01
2    04/01/17 02
3    04/01/17 03
4    04/01/17 04
Name: DATETIME, dtype: str


In [4]:
# attacks
df_train_attacks.columns = df_train_attacks.columns.str.strip()
df_train_attacks["DATETIME"] = pd.to_datetime(df_train_attacks["DATETIME"], format="%d/%m/%y %H")
df_train_attacks = df_train_attacks.set_index("DATETIME").sort_index()

print("attacks\n", df_train_attacks.index.min(), df_train_attacks.index.max())

attacks
 2016-07-04 00:00:00 2016-12-25 00:00:00


In [5]:
# normal
df_train_normal.columns = df_train_normal.columns.str.strip()
df_train_normal["DATETIME"] = pd.to_datetime(df_train_normal["DATETIME"], format="%d/%m/%y %H")
df_train_normal = df_train_normal.set_index("DATETIME").sort_index()

print("normal\n", df_train_normal.index.min(), df_train_normal.index.max())

normal
 2014-01-06 00:00:00 2015-01-06 00:00:00


In [6]:
# test
df_test.columns = df_test.columns.str.strip()
df_test["DATETIME"] = pd.to_datetime(df_test["DATETIME"], format="%d/%m/%y %H")
df_test = df_test.set_index("DATETIME").sort_index()

print("test\n", df_test.index.min(), df_test.index.max())

test
 2017-01-04 00:00:00 2017-04-01 00:00:00


In [7]:
# sensor columns must match across files
assert list(df_test.columns) == list(df_train_attacks.columns.drop("ATT_FLAG"))
assert list(df_test.columns) == list(df_train_normal.columns.drop("ATT_FLAG"))

In [8]:
# label values per file
print("normal\n", df_train_normal["ATT_FLAG"].value_counts())
print("attacks\n", df_train_attacks["ATT_FLAG"].value_counts())

normal
 ATT_FLAG
0    8761
Name: count, dtype: int64
attacks
 ATT_FLAG
-999    3958
 1       219
Name: count, dtype: int64


In [9]:
# start and end of each labeled attack run
is_attack = df_train_attacks["ATT_FLAG"] == 1
run_id = (is_attack != is_attack.shift()).cumsum()
runs = is_attack[is_attack].index.to_series().groupby(run_id[is_attack]).agg(["min", "max"])

print(runs)

                         min                 max
ATT_FLAG                                        
2        2016-09-14 03:00:00 2016-09-15 20:00:00
4        2016-10-09 09:00:00 2016-10-11 20:00:00
6        2016-10-30 19:00:00 2016-11-01 07:00:00
8        2016-11-27 04:00:00 2016-11-27 10:00:00
10       2016-12-06 21:00:00 2016-12-09 21:00:00


In [10]:
# hour spacing per file; a single 1 hour row means gaps and duplicates does not exist
print("normal\n", df_train_normal.index.to_series().diff().value_counts())
print("attacks\n", df_train_attacks.index.to_series().diff().value_counts())
print("test\n", df_test.index.to_series().diff().value_counts())

normal
 DATETIME
0 days 01:00:00    8760
Name: count, dtype: int64
attacks
 DATETIME
0 days 01:00:00    4176
Name: count, dtype: int64
test
 DATETIME
0 days 01:00:00    2088
Name: count, dtype: int64


In [11]:
# verify src/data.py against the EDA results
from src.data import load_csv, get_features, get_labels

normal = load_csv("data/train_normal.csv")
attacks = load_csv("data/train_attacks.csv")
test = load_csv("data/test.csv")

assert get_labels(attacks).sum() == 219
assert get_labels(normal).sum() == 0
assert get_features(normal).shape[1] == 43
assert get_features(attacks).shape[1] == 43
assert get_features(test).shape[1] == 43

try:
    get_labels(test)
    print("Fail: test labels did not raise.")

except KeyError:
    print("Pass: all checks.")

Pass: all checks.
